# Excel Lokomotif 2019-2026 -> Pandas -> SQLite

Notebook ini membaca **seluruh workbook** `DAFTAR NOMOR EQUIPMENT LOKOMOTIF` dari tahun **2019 sampai 2026**, lalu menormalisasi isinya ke SQLite.

File sumber berbentuk **report layout**, bukan tabel datar. Dalam satu worksheet terdapat dua blok lokomotif secara horizontal:

```text
Blok 1: A:G
Blok 2: I:O
```

Setiap blok berisi metadata (`NO.SERI LOKOMOTIF`, `DIPO INDUK`, `JENIS PERAWATAN`, `PROGRAM BULAN`, `MASUK`, `KELUAR`) dan tabel detail komponen 7 kolom.

## Yang berubah dibanding versi sebelumnya

1. **Multi tahun** - seluruh file 2019 s/d 2026 diproses dalam satu run, bukan satu file saja.
2. **Referensi asal di setiap row** - tabel lokomotif maupun tabel komponen menyimpan `source_file`, `source_year`, `source_sheet`, dan `block_index`.
3. **Block template dibuang** - block tanpa nomor lokomotif berarti form yang belum dipakai mencatat perawatan, jadi tidak ikut masuk database. Jumlahnya tetap tercatat pada `sheet_availability`.
4. **Tanggal dilengkapi dari PROGRAM BULAN** - kalau `MASUK` / `KELUAR` kosong, tanggal diturunkan dari bulan programnya: masuk tanggal 1, keluar tanggal akhir bulan. Asalnya ditandai lewat `masuk_source` / `keluar_source`.
5. **Tabel komponen membawa identitas perawatan** - nomor lokomotif, tanggal masuk, tanggal keluar, dan `tahun_maintenance`, sehingga bisa dianalisis tanpa join ke tabel event.
6. **Data availability** - hasil scan per file / per sheet / per block disimpan ke tabel `sheet_availability`.
7. **Varian label antar tahun** - 2019-2021 memakai `NO. SERI LOKO`, 2020 memakai `MSK` dan `SELESAI`. Semua varian dipetakan ke field yang sama.
8. **Perbaikan bug metadata** - `DIPO INDUK` sebelumnya terbaca `"INDUK"` (potongan label di kolom B), dan nilai berprefiks `": SDT"` tidak dibersihkan.
9. **Baris legenda dibuang** - baris `= DISMANTLE`, `= INSTALL`, `= PENAMBAHAN` di bawah tabel tidak lagi ikut terbaca sebagai komponen.

Target normalisasi:

```text
maintenance_events
        |
        | 1:N
        v
equipment_components
```

## 1. Import library

In [55]:
# Uncomment jika belum terinstall:
# %pip install pandas openpyxl

import calendar
import re
import shutil
import sqlite3
import time

import numpy as np
import pandas as pd

from pathlib import Path
from datetime import date, datetime, timezone

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

print("Pandas:", pd.__version__)
print("SQLite:", sqlite3.sqlite_version)

Pandas: 2.2.2
SQLite: 3.45.3


## 2. Configuration

`EXCEL_FILE` tunggal diganti `FILE_PATTERN`, sehingga seluruh workbook yang cocok akan diproses.

`SCHEMA_VERSION = 2` karena tabel sekarang menyimpan kolom referensi Excel. Jika database masih memakai schema v1, database akan di-backup lebih dulu lalu dibangun ulang - seluruh isinya berasal dari Excel sehingga aman di-regenerate.

In [56]:
DATA_DIR = Path(".")
FILE_PATTERN = "Daftar Nomor Equipment Lokomotif *.xlsx"

DB_FILE = Path("kai.db")
DEBUG_OUTPUT = Path("parsed_lokomotif_debug.xlsx")

# Lebar tabel detail komponen: NO, NAMA, ASAL x2, PENGGANTI x2, KET
BLOCK_WIDTH = 7

# Metadata selalu berada pada baris atas block
METADATA_SEARCH_ROWS = 20

SCHEMA_VERSION = 3
REBUILD_DB_IF_SCHEMA_OUTDATED = True

# None = semua worksheet pada setiap file
SHEETS_TO_PROCESS = None

print("Data dir :", DATA_DIR.resolve())
print("DB       :", DB_FILE)

Data dir : C:\Users\bnurhuda\Documents\personal\KAI
DB       : kai.db


## 3. Discovery file 2019-2026

In [57]:
def extract_year(file_name):
    match = re.search(r"(20\d{2})", str(file_name))

    if match is None:
        return None

    return int(match.group(1))


def discover_source_files(
    data_dir=DATA_DIR,
    pattern=FILE_PATTERN
):
    return [
        path
        for path in sorted(data_dir.glob(pattern))
        if not path.name.startswith("~$")
    ]


EXCEL_FILES = discover_source_files()

file_catalog = pd.DataFrame([
    {
        "source_file": path.name,
        "source_year": extract_year(path.name),
        "size_kb": round(path.stat().st_size / 1024),
    }
    for path in EXCEL_FILES
])

print("File ditemukan:", len(EXCEL_FILES))

display(file_catalog)

File ditemukan: 8


,source_file,source_year,size_kb
0,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,347
1,Daftar Nomor Equipment Lokomotif 2020.xlsx,2020,342
2,Daftar Nomor Equipment Lokomotif 2021.xlsx,2021,395
3,Daftar Nomor Equipment Lokomotif 2022.xlsx,2022,557
4,Daftar Nomor Equipment Lokomotif 2023.xlsx,2023,567
5,Daftar Nomor Equipment Lokomotif 2024.xlsx,2024,726
6,Daftar Nomor Equipment Lokomotif 2025.xlsx,2025,698
7,Daftar Nomor Equipment Lokomotif 2026.xlsx,2026,619


### 3.1 Cek tahun yang hilang

Bagian pertama dari *data availability*: apakah ada tahun pada rentang target yang filenya memang tidak ada.

Catatan: file `~$...xlsx` adalah file lock Excel (muncul saat workbook sedang dibuka) dan sengaja diabaikan karena bukan workbook yang valid.

In [58]:
EXPECTED_YEARS = list(range(2019, 2027))

found_years = sorted(
    int(y)
    for y in file_catalog["source_year"].dropna().unique()
)

missing_years = [
    y
    for y in EXPECTED_YEARS
    if y not in found_years
]

print("Tahun tersedia :", found_years)
print("Tahun hilang   :", missing_years if missing_years else "-")

if not EXCEL_FILES:
    raise FileNotFoundError(
        f"Tidak ada file cocok pola {FILE_PATTERN!r} di {DATA_DIR.resolve()}"
    )

Tahun tersedia : [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Tahun hilang   : -


## 4. Inventory worksheet

Membuka setiap workbook satu kali untuk mendata sheet-nya.

Nama sheet tidak konsisten antar tahun (`Sheet1 (3)`, `Sheet  (34)`, `1340 1350`), sehingga nama sheet **tidak dipakai** sebagai identitas lokomotif. Identitas diambil dari isi cell; nama sheet hanya disimpan sebagai referensi lokasi.

In [59]:
def sheet_inventory(files):
    rows = []

    for path in files:
        xls = pd.ExcelFile(path, engine="openpyxl")

        for index, sheet in enumerate(xls.sheet_names, start=1):
            rows.append({
                "source_file": path.name,
                "source_year": extract_year(path.name),
                "sheet_index": index,
                "source_sheet": sheet,
            })

    return pd.DataFrame(rows)


sheet_catalog = sheet_inventory(EXCEL_FILES)

print("Total worksheet:", len(sheet_catalog))

display(
    sheet_catalog
    .groupby(["source_year", "source_file"])
    .size()
    .reset_index(name="sheet_count")
)

Total worksheet: 416


,source_year,source_file,sheet_count
0,2019,Daftar Nomor Equipment Lokomotif 2019.xlsx,36
1,2020,Daftar Nomor Equipment Lokomotif 2020.xlsx,34
2,2021,Daftar Nomor Equipment Lokomotif 2021.xlsx,41
3,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,55
4,2023,Daftar Nomor Equipment Lokomotif 2023.xlsx,56
5,2024,Daftar Nomor Equipment Lokomotif 2024.xlsx,71
6,2025,Daftar Nomor Equipment Lokomotif 2025.xlsx,69
7,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,54


# PART A - Raw grid dan referensi Excel

Worksheet dibaca dengan `header=None` supaya posisi cell tetap utuh.

Konsekuensinya index baris pandas mulai dari 0 sedangkan Excel dari 1. Semua referensi yang disimpan ke database sudah dikonversi ke penomoran Excel, jadi bisa langsung dipakai membuka file aslinya.

## 5. Preview raw grid

In [60]:
sample_file = EXCEL_FILES[-1]
sample_xls = pd.ExcelFile(sample_file, engine="openpyxl")
sample_sheet = sample_xls.sheet_names[0]

raw = pd.read_excel(
    sample_xls,
    sheet_name=sample_sheet,
    header=None,
    dtype=object,
    engine="openpyxl"
)

print("File :", sample_file.name)
print("Sheet:", sample_sheet)
print("Shape:", raw.shape)

display(raw.head(12))

File : Daftar Nomor Equipment Lokomotif 2026.xlsx
Sheet: 8302 9209
Shape: (94, 15)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,DAFTAR NOMOR EQUIPMENT LOKOMOTIF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,DAFTAR NOMOR EQUIPMENT LOKOMOTIF,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NO.SERI LOKOMOTIF,NaN,CC 201 83 02,0WC,1008666,MASUK,2026-01-07 00:00:00,NaN,NO. SERI LOKOMOTIF,NaN,CC 201 92 09,0GB,1009525,MASUK,2026-01-08 00:00:00
3,DIPO INDUK,INDUK,YK,NaN,1159486,KELUAR,2026-01-26 00:00:00,NaN,DIPO INDUK,NaN,BD,NaN,1404656,KELUAR,2026-02-01 00:00:00
4,JENIS PERAWATAN,NaN,P48,NaN,1159499,NaN,NaN,NaN,JENIS PERAWATAN,NaN,P48,NaN,1404657,NaN,NaN
5,PROGRAM BULAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PROGRAM BULAN,NaN,NaN,NaN,NaN,NaN,NaN
6,NO.,NAMA KOMPONEN,KOMPONEN ASAL,NaN,KOMPONEN PENGGANTI,NaN,KET.,NaN,NO.,NAMA KOMPONEN,KOMPONEN ASAL,NaN,KOMPONEN PENGGANTI,NaN,KET.
7,NaN,NaN,KODE CETAK,NO.MANUF,KODE CETAK,NO.MANUF,NaN,NaN,NaN,NaN,KODE CETAK,NO.MANUF,KODE CETAK,NO.MANUF,NaN
8,1,MOTOR DIESEL,MD-1151907,292802,NaN,NaN,Tidak turun,NaN,1,MOTOR DIESEL,MD-1159434,299245,NaN,NaN,Tidak turun
9,2,CYLINDER ASSY 1R,CA-1163500,LG011110907-MG20030564,NaN,NaN,Tidak turun,NaN,2,CYLINDER ASSY 1R,CA-1152664,LG14050401-MG20061401,NaN,NaN,Tidak turun


## 6. Helper cleaning cell

In [61]:
def clean_cell(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if isinstance(value, str):
        value = value.replace("\n", " ")
        value = re.sub(r"\s+", " ", value).strip()

        if value == "":
            return None

    return value


def normalize_label(value):
    value = clean_cell(value)

    if value is None:
        return ""

    text = str(value).upper().strip()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def strip_leading_colon(value):
    """Nilai metadata sering ditulis ": SDT" dalam satu cell."""
    if isinstance(value, str):
        value = re.sub(r"^\s*:\s*", "", value).strip()

        if value == "":
            return None

    return value

## 7. Helper kolom Excel

Referensi yang disimpan ke database cukup sampai tingkat **file, tahun, sheet, dan block**. Alamat cell tidak ikut disimpan.

Helper kecil di bawah ini hanya dipakai untuk keperluan tampilan saat memeriksa struktur worksheet, supaya posisi block terbaca sebagai `A:G` dan `I:O` alih-alih index angka.

In [62]:
def excel_col_letter(col_idx):
    """0 -> A, 8 -> I"""
    letters = ""
    n = col_idx + 1

    while n > 0:
        n, remainder = divmod(n - 1, 26)
        letters = chr(65 + remainder) + letters

    return letters


print("block 1 mulai kolom", excel_col_letter(0))
print("block 2 mulai kolom", excel_col_letter(8))

block 1 mulai kolom A
block 2 mulai kolom I


## 8. Deteksi posisi block

Block dicari lewat marker `NAMA KOMPONEN` yang selalu berada di kolom kedua tiap block:

```python
block_start = nama_komponen_col - 1
```

Hasil pemindaian 416 worksheet: seluruhnya memakai pola block start `(0, 8)`. Deteksi tetap dibuat dinamis agar tahan terhadap pergeseran kolom.

In [63]:
def detect_block_start_columns(df):
    starts = []

    for r in range(df.shape[0]):
        for c in range(df.shape[1]):
            if normalize_label(df.iat[r, c]) == "NAMA KOMPONEN":
                start_col = c - 1

                if start_col >= 0:
                    starts.append(start_col)

    return sorted(set(starts))


block_starts = detect_block_start_columns(raw)

print("Detected block starts:", block_starts)
print("Kolom Excel          :", [excel_col_letter(c) for c in block_starts])

Detected block starts: [0, 8]
Kolom Excel          : ['A', 'I']


## 9. Preview setiap block

In [64]:
for i, start_col in enumerate(block_starts, start=1):
    print("=" * 80)
    print(
        f"BLOCK {i} | kolom "
        f"{excel_col_letter(start_col)}:"
        f"{excel_col_letter(start_col + BLOCK_WIDTH - 1)}"
    )

    display(
        raw.iloc[:12, start_col:start_col + BLOCK_WIDTH]
    )

BLOCK 1 | kolom A:G


,0,1,2,3,4,5,6
0,DAFTAR NOMOR EQUIPMENT LOKOMOTIF,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NO.SERI LOKOMOTIF,NaN,CC 201 83 02,0WC,1008666,MASUK,2026-01-07 00:00:00
3,DIPO INDUK,INDUK,YK,NaN,1159486,KELUAR,2026-01-26 00:00:00
4,JENIS PERAWATAN,NaN,P48,NaN,1159499,NaN,NaN
5,PROGRAM BULAN,NaN,NaN,NaN,NaN,NaN,NaN
6,NO.,NAMA KOMPONEN,KOMPONEN ASAL,NaN,KOMPONEN PENGGANTI,NaN,KET.
7,NaN,NaN,KODE CETAK,NO.MANUF,KODE CETAK,NO.MANUF,NaN
8,1,MOTOR DIESEL,MD-1151907,292802,NaN,NaN,Tidak turun
9,2,CYLINDER ASSY 1R,CA-1163500,LG011110907-MG20030564,NaN,NaN,Tidak turun


BLOCK 2 | kolom I:O


,8,9,10,11,12,13,14
0,DAFTAR NOMOR EQUIPMENT LOKOMOTIF,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NO. SERI LOKOMOTIF,NaN,CC 201 92 09,0GB,1009525,MASUK,2026-01-08 00:00:00
3,DIPO INDUK,NaN,BD,NaN,1404656,KELUAR,2026-02-01 00:00:00
4,JENIS PERAWATAN,NaN,P48,NaN,1404657,NaN,NaN
5,PROGRAM BULAN,NaN,NaN,NaN,NaN,NaN,NaN
6,NO.,NAMA KOMPONEN,KOMPONEN ASAL,NaN,KOMPONEN PENGGANTI,NaN,KET.
7,NaN,NaN,KODE CETAK,NO.MANUF,KODE CETAK,NO.MANUF,NaN
8,1,MOTOR DIESEL,MD-1159434,299245,NaN,NaN,Tidak turun
9,2,CYLINDER ASSY 1R,CA-1152664,LG14050401-MG20061401,NaN,NaN,Tidak turun


# PART B - Parse metadata lokomotif

Metadata dicari berdasarkan label, bukan nomor row absolut.

## 10. Label dan variannya antar tahun

Hasil pemindaian seluruh file menunjukkan label tidak konsisten antar tahun:

| Field | Varian yang ditemukan | Tahun |
|---|---|---|
| `no_seri_lokomotif` | `NO.SERI LOKOMOTIF` | 2021-2026 |
| | `NO. SERI LOKO` | 2019-2021 |
| `masuk` | `MASUK` | 2020-2026 |
| | `MSK` | 2020-2021 |
| | `MASUK BY` | 2020 |
| `keluar` | `KELUAR` | 2020-2026 |
| | `KELUAR BY`, `SELESAI` | 2020 |
| `ganti_di_dipo` | `GANTI DI DIPO` | 2023 |

File 2019 memang tidak memiliki kolom `MASUK` / `KELUAR` sama sekali, jadi kedua field itu akan `NULL` untuk seluruh row tahun 2019. Ini kondisi sumber, bukan kegagalan parser.

In [65]:
METADATA_LABELS = {
    "no_seri_lokomotif": [
        "NO.SERI LOKOMOTIF",
        "NO. SERI LOKOMOTIF",
        "NO SERI LOKOMOTIF",
        "NO.SERI LOKO",
        "NO. SERI LOKO",
        "NO SERI LOKO",
    ],
    "dipo_induk": ["DIPO INDUK"],
    "jenis_perawatan": ["JENIS PERAWATAN"],
    "program_bulan": ["PROGRAM BULAN"],
    "masuk": ["MASUK", "MASUK BY", "MSK"],
    "keluar": ["KELUAR", "KELUAR BY", "SELESAI"],
    "ganti_di_dipo": ["GANTI DI DIPO"],
}

# Dipakai agar teks label tidak pernah dianggap sebagai nilai
ALL_LABEL_FORMS = {
    normalize_label(variant)
    for variants in METADATA_LABELS.values()
    for variant in variants
} | {
    "NO",
    "NAMA KOMPONEN",
    "KOMPONEN ASAL",
    "KOMPONEN PENGGANTI",
    "KET",
    "KODE CETAK",
    "NO MANUF",
}

print(len(ALL_LABEL_FORMS), "bentuk label dikenali")

19 bentuk label dikenali


## 11. Mengambil nilai di kanan label

Ini bagian yang paling banyak diperbaiki. Pada file 2022-2026 baris `DIPO INDUK` berbentuk:

```text
A3 = "DIPO INDUK "    B3 = "INDUK"    C3 = ": SDT"
```

Versi lama mengambil cell pertama yang tidak kosong, sehingga menghasilkan `"INDUK"` - potongan label di kolom B - bukan `SDT`. Akibatnya kolom `dipo_induk` pada database lama berisi nilai yang sama untuk hampir seluruh row.

Aturan sekarang:

1. lewati cell kosong,
2. buang prefiks `":"` (`": SDT"` -> `"SDT"`),
3. lewati cell yang isinya hanya potongan dari label itu sendiri (`INDUK` dari `DIPO INDUK`),
4. lewati cell yang ternyata label lain (`MASUK`, `KET`, dan seterusnya).

In [66]:
def label_matches(value, variants):
    normalized = normalize_label(value)

    if normalized == "":
        return False

    return normalized in {
        normalize_label(v)
        for v in variants
    }


def first_value_to_right(
    df,
    row_idx,
    label_col,
    block_end,
    label_text
):
    label_tokens = set(
        normalize_label(label_text).split()
    )

    for c in range(label_col + 1, block_end + 1):
        value = strip_leading_colon(
            clean_cell(df.iat[row_idx, c])
        )

        if value is None:
            continue

        normalized = normalize_label(value)

        # "INDUK" adalah pecahan label "DIPO INDUK", bukan nilai
        if normalized and set(normalized.split()) <= label_tokens:
            continue

        if normalized in ALL_LABEL_FORMS:
            continue

        return value

    return None

## 12. parse_metadata

Selain nilai, fungsi ini mengembalikan baris tempat setiap label ditemukan, sehingga referensi cell metadata bisa ikut disimpan.

In [67]:
def parse_metadata(
    df,
    block_start,
    block_width=BLOCK_WIDTH,
    search_rows=METADATA_SEARCH_ROWS
):
    block_end = min(
        block_start + block_width - 1,
        df.shape[1] - 1
    )

    result = {key: None for key in METADATA_LABELS}
    refs = {key: None for key in METADATA_LABELS}

    max_row = min(search_rows, df.shape[0])

    for r in range(max_row):
        for c in range(block_start, block_end + 1):
            cell = df.iat[r, c]

            for key, variants in METADATA_LABELS.items():
                if result[key] is not None:
                    continue

                if not label_matches(cell, variants):
                    continue

                value = first_value_to_right(
                    df,
                    r,
                    c,
                    block_end,
                    str(clean_cell(cell))
                )

                if value is not None:
                    result[key] = value
                    refs[key] = r

    return result, refs

## 13. Normalisasi nomor lokomotif

Penulisan nomor seri berbeda-beda antar tahun:

```text
"CC 203 98 08 (20)"
": CC 206 13 40"
"CC 201 83 02"
```

Agar satu lokomotif dapat dilacak lintas tahun, disimpan tiga bentuk:

- `no_seri_lokomotif` - nilai mentah apa adanya (audit trail),
- `lokomotif_no` - bentuk tampilan yang sudah dibersihkan,
- `lokomotif_key` - hanya digit, dipakai sebagai kunci join antar tahun.

In [68]:
def normalize_loco(value):
    value = clean_cell(value)

    if value is None:
        return None, None

    text = re.sub(r"\([^)]*\)", "", str(value)).upper()
    text = strip_leading_colon(text) or ""
    text = re.sub(r"[^A-Z0-9]+", " ", text).strip()
    text = re.sub(r"\s+", " ", text)

    if text == "":
        return None, None

    digits = re.sub(r"\D", "", text)

    return text, (digits or None)


for sample in ["CC 203 98 08 (20)", ": CC 206 13 40", "CC 201 83 02"]:
    print(f"{sample:22} -> {normalize_loco(sample)}")

CC 203 98 08 (20)      -> ('CC 203 98 08', '2039808')
: CC 206 13 40         -> ('CC 206 13 40', '2061340')
CC 201 83 02           -> ('CC 201 83 02', '2018302')


## 14. Parse tanggal

Sumber memakai format Indonesia `DD-MM-YYYY` untuk teks, sebagian lain sudah berupa datetime asli Excel. `dayfirst=True` menangani keduanya.

In [69]:
def parse_date(value):
    value = clean_cell(value)

    if value is None:
        return None

    parsed = pd.to_datetime(
        value,
        errors="coerce",
        dayfirst=True
    )

    if pd.isna(parsed):
        return None

    return parsed.date()


print(
    parse_date("19-11-2020"),
    parse_date(datetime(2026, 1, 7)),
    parse_date("-")
)

2020-11-19 2026-01-07 None


## 15. Test metadata lintas tahun

Uji cepat pada satu sheet dari setiap tahun untuk memastikan varian label tertangani.

In [70]:
for path in EXCEL_FILES:
    xls = pd.ExcelFile(path, engine="openpyxl")
    sheet = xls.sheet_names[0]

    df = pd.read_excel(
        xls,
        sheet_name=sheet,
        header=None,
        dtype=object,
        engine="openpyxl"
    )

    starts = detect_block_start_columns(df)

    if not starts:
        print(f"{extract_year(path.name)} | {sheet:16} | tidak ada block")
        continue

    metadata, _ = parse_metadata(df, starts[0])

    print(
        f"{extract_year(path.name)} | {sheet:16} | "
        f"seri={metadata['no_seri_lokomotif']} | "
        f"dipo={metadata['dipo_induk']} | "
        f"rawat={metadata['jenis_perawatan']} | "
        f"masuk={metadata['masuk']} | "
        f"keluar={metadata['keluar']}"
    )

2019 | Sheet1           | seri=CC 203 98 08 (20) | dipo=JNG | rawat=SPA 2 / P48 | masuk=None | keluar=None
2020 | Sheet  (34)      | seri=CC 206 13 25 | dipo=SDT | rawat=GO | masuk=19-11-2020 | keluar=18-12-2020
2021 | sheet (2)        | seri=None | dipo=None | rawat=None | masuk=None | keluar=None
2022 | 1340 1350        | seri=CC 206 13 40 | dipo=SDT | rawat=GO | masuk=2022-01-06 00:00:00 | keluar=2022-04-01 00:00:00
2023 | Sheet2 (16)      | seri=None | dipo=None | rawat=None | masuk=None | keluar=None
2024 | 9209 8302        | seri=CC 201 92 09 | dipo=BD | rawat=P.24 | masuk=2024-01-12 00:00:00 | keluar=2024-02-01 00:00:00
2025 | 1333 1330        | seri=CC 206 13 33 | dipo=CN | rawat=P48+MO | masuk=2025-01-09 00:00:00 | keluar=2025-01-22 00:00:00
2026 | 8302 9209        | seri=CC 201 83 02 | dipo=YK | rawat=P48 | masuk=2026-01-07 00:00:00 | keluar=2026-01-26 00:00:00


# PART C - Parse detail komponen

Struktur 7 kolom pada setiap block:

```text
0 NO.
1 NAMA KOMPONEN
2 ASAL - KODE CETAK
3 ASAL - NO.MANUF
4 PENGGANTI - KODE CETAK
5 PENGGANTI - NO.MANUF
6 KET.
```

In [71]:
COMPONENT_COLUMNS = [
    "component_no",
    "component_name",
    "asal_kode_cetak",
    "asal_no_manuf",
    "pengganti_kode_cetak",
    "pengganti_no_manuf",
    "keterangan",
]

COMPONENT_DETAIL_COLUMNS = [
    "asal_kode_cetak",
    "asal_no_manuf",
    "pengganti_kode_cetak",
    "pengganti_no_manuf",
    "keterangan",
]

## 16. Find header row

In [72]:
def find_component_header_row(
    df,
    block_start,
    block_width=BLOCK_WIDTH
):
    block_end = min(
        block_start + block_width,
        df.shape[1]
    )

    for r in range(df.shape[0]):
        for c in range(block_start, block_end):
            if normalize_label(df.iat[r, c]) == "NAMA KOMPONEN":
                return r

    return None

## 17. Batas bawah tabel

Di bawah tabel komponen terdapat baris legenda, contohnya pada file 2024 dan 2026:

```text
= Sudah diinstal, belum ditempel
= DISMANTLE
= INSTALL
= PENAMBAHAN
```

Versi lama ikut membaca baris ini sebagai komponen, dan karena `component_no` / `component_name` di-`ffill()`, baris legenda tersebut mewarisi nama komponen terakhir - menghasilkan row palsu.

Sekarang pembacaan block dihentikan begitu bertemu baris legenda (baris yang memuat cell diawali `=`).

In [73]:
def row_is_empty(values):
    return all(v is None for v in values)


def row_is_legend(values):
    texts = [
        str(v).strip()
        for v in values
        if v is not None
    ]

    return bool(texts) and any(
        t.startswith("=")
        for t in texts
    )


def row_is_header_repeat(values):
    normalized = {
        normalize_label(v)
        for v in values
    }

    return bool(
        normalized & {"NAMA KOMPONEN", "KODE CETAK", "NO MANUF"}
    )

## 18. parse_components

Merged cell vertikal (`INJECTION NOZZLE` misalnya) dibaca pandas sebagai:

```text
8    INJECTION NOZZLE
NaN  NaN
NaN  NaN
```

sehingga `component_no` dan `component_name` perlu `ffill()`.

Row disaring dengan syarat minimal satu field equipment atau keterangan terisi. Perhatikan bahwa syarat ini **tidak** mengharuskan komponen asal ada: row yang hanya berisi komponen pengganti adalah pemasangan baru dan tetap disimpan.

Urutan row hasil parsing mengikuti urutan barisnya di worksheet.

In [74]:
def parse_components(
    df,
    block_start,
    block_width=BLOCK_WIDTH
):
    header_row = find_component_header_row(
        df,
        block_start,
        block_width
    )

    if header_row is None:
        return pd.DataFrame(columns=COMPONENT_COLUMNS), None, 0

    # Dua tingkat header: row utama + row subheader
    data_start = header_row + 2

    rows = []

    for r in range(data_start, df.shape[0]):
        values = [
            clean_cell(df.iat[r, c]) if c < df.shape[1] else None
            for c in range(block_start, block_start + block_width)
        ]

        if row_is_legend(values):
            break

        if row_is_empty(values):
            continue

        if row_is_header_repeat(values):
            continue

        rows.append(values)

    result = pd.DataFrame(rows, columns=COMPONENT_COLUMNS)

    raw_row_count = len(result)

    if result.empty:
        return result, header_row, raw_row_count

    for col in ("component_no", "component_name"):
        result[col] = (
            result[col]
            .replace("", np.nan)
            .ffill()
        )

    # Minimal satu field equipment / keterangan harus terisi,
    # supaya baris template kosong tidak ikut masuk
    result = result[
        result[COMPONENT_DETAIL_COLUMNS]
        .notna()
        .any(axis=1)
    ].reset_index(drop=True)

    return result, header_row, raw_row_count

## 19. Test component parsing

In [75]:
for i, start_col in enumerate(block_starts, start=1):
    components, header_row, raw_count = parse_components(
        raw,
        start_col,
        BLOCK_WIDTH
    )

    baru = components[
        components["asal_kode_cetak"].isna()
        & components["asal_no_manuf"].isna()
        & (
            components["pengganti_kode_cetak"].notna()
            | components["pengganti_no_manuf"].notna()
        )
    ]

    print("=" * 80)
    print(
        f"BLOCK {i} | header row Excel = {header_row + 1} | "
        f"row terbaca = {raw_count} | row terpakai = {len(components)} | "
        f"pemasangan baru = {len(baru)}"
    )

    display(components.head(8))

BLOCK 1 | header row Excel = 7 | row terbaca = 80 | row terpakai = 76 | pemasangan baru = 6


,component_no,component_name,asal_kode_cetak,asal_no_manuf,pengganti_kode_cetak,pengganti_no_manuf,keterangan
0,1.0,MOTOR DIESEL,MD-1151907,292802,None,None,Tidak turun
1,2.0,CYLINDER ASSY 1R,CA-1163500,LG011110907-MG20030564,None,None,Tidak turun
2,2.0,2R,CA-1163488,LG02050779-MG19120717,None,None,Tidak turun
3,2.0,3R,CA-1444632,LG19070825-MG15021754,None,None,Tidak turun
4,2.0,4R,CA-1163512,CG95030278-MG20080660,None,None,Tidak turun
5,2.0,1L,CA-1163484,CG96050001-MG20080661,None,None,Tidak turun
6,2.0,2L,CA-1163492,C8304082-MG20080676,None,None,Tidak turun
7,2.0,3L,CA-1163504,E7609479-MG20080637,None,None,Tidak turun


BLOCK 2 | header row Excel = 7 | row terbaca = 80 | row terpakai = 76 | pemasangan baru = 6


,component_no,component_name,asal_kode_cetak,asal_no_manuf,pengganti_kode_cetak,pengganti_no_manuf,keterangan
0,1.0,MOTOR DIESEL,MD-1159434,299245,None,None,Tidak turun
1,2.0,CYLINDER ASSY 1R,CA-1152664,LG14050401-MG20061401,None,None,Tidak turun
2,2.0,2R,CA-1152649,LG14050391-MG20090277,None,None,Tidak turun
3,2.0,3R,CA-1152609,LG14050424-MG20030580,None,None,Tidak turun
4,2.0,4R,CA-1152625,LG14050395-MG20090210,None,None,Tidak turun
5,2.0,1L,CA-1404619,C8211077-MG20090278,None,None,Tidak turun
6,2.0,2L,CA-1404617,CG91080171-MG20090208,None,None,Tidak turun
7,2.0,3L,CA-1404614,CG97040427-MG20030585,None,None,Tidak turun


# PART D - Block menjadi parent + children

Setiap block menghasilkan satu row `maintenance_events` dan sejumlah row `equipment_components`.

Referensi asal yang disimpan pada **kedua** tabel:

```text
source_file   -> Daftar Nomor Equipment Lokomotif 2026.xlsx
source_year   -> 2026
source_sheet  -> 8302 9209
block_index   -> 1        (block 1 = kolom A:G, block 2 = kolom I:O)
```

## 20. Block template

Sebagian block hanya berisi kerangka form: nomor dan nama komponen sudah tercetak, tetapi belum ada nomor lokomotif dan belum ada satupun nomor equipment. Block seperti ini belum dipakai untuk pencatatan perawatan, jadi **tidak dimasukkan** ke database - baik row event-nya maupun row komponennya.

Penanda yang dipakai adalah ketiadaan nomor lokomotif. Kalau nomor lokomotif ada, block tetap disimpan meskipun sebagian besar isinya kosong.

Jumlah block template per file tetap dicatat di `sheet_availability`, sehingga informasinya tidak hilang.

In [76]:
def is_template_block(no_seri, component_count):
    """Block tanpa nomor lokomotif = form kosong, belum dipakai."""
    if no_seri is not None:
        return False

    return True

## 21. parse_locomotive_block

In [77]:
def parse_locomotive_block(
    df,
    source_file,
    source_sheet,
    sheet_index,
    block_index,
    block_start,
    block_width=BLOCK_WIDTH
):
    metadata, _ = parse_metadata(
        df,
        block_start,
        block_width
    )

    components, header_row, raw_row_count = parse_components(
        df,
        block_start,
        block_width
    )

    lokomotif_no, lokomotif_key = normalize_loco(
        metadata["no_seri_lokomotif"]
    )

    event = {
        "source_file": source_file,
        "source_year": extract_year(source_file),
        "source_sheet": source_sheet,
        "sheet_index": sheet_index,
        "block_index": block_index,

        "no_seri_lokomotif": metadata["no_seri_lokomotif"],
        "lokomotif_no": lokomotif_no,
        "lokomotif_key": lokomotif_key,
        "dipo_induk": metadata["dipo_induk"],
        "jenis_perawatan": metadata["jenis_perawatan"],
        "program_bulan": metadata["program_bulan"],
        "ganti_di_dipo": metadata["ganti_di_dipo"],
        "masuk": parse_date(metadata["masuk"]),
        "keluar": parse_date(metadata["keluar"]),

        "component_count": len(components),
        "component_rows_scanned": raw_row_count,
        "is_template": is_template_block(
            metadata["no_seri_lokomotif"],
            len(components)
        ),
    }

    if not components.empty:
        components = components.copy()
        components.insert(0, "block_index", block_index)
        components.insert(0, "source_sheet", source_sheet)
        components.insert(0, "source_year", extract_year(source_file))
        components.insert(0, "source_file", source_file)

    return event, components

## 22. Parse satu worksheet

Perhatikan `xls` yang dikirim sebagai parameter. Versi sebelumnya memanggil `pd.read_excel(path, sheet_name=...)` per sheet, artinya workbook di-parse ulang dari awal untuk setiap sheet. Dengan 416 worksheet biayanya sangat besar. Membuka `pd.ExcelFile` sekali per file mempercepat proses menjadi sekitar satu menit untuk seluruh 2019-2026.

In [78]:
def parse_sheet(
    xls,
    source_file,
    sheet_name,
    sheet_index,
    block_width=BLOCK_WIDTH
):
    df = pd.read_excel(
        xls,
        sheet_name=sheet_name,
        header=None,
        dtype=object,
        engine="openpyxl"
    )

    starts = detect_block_start_columns(df)

    events = []
    frames = []

    for block_index, start_col in enumerate(starts, start=1):
        event, components = parse_locomotive_block(
            df=df,
            source_file=source_file,
            source_sheet=sheet_name,
            sheet_index=sheet_index,
            block_index=block_index,
            block_start=start_col,
            block_width=block_width
        )

        events.append(event)

        if not components.empty:
            frames.append(components)

    events_df = pd.DataFrame(events)

    components_df = (
        pd.concat(frames, ignore_index=True)
        if frames
        else pd.DataFrame()
    )

    return events_df, components_df

## 23. Parse satu workbook

Block template dibuang di sini, bersama komponen yang menempel padanya. Komponen disaring lewat kombinasi `(source_sheet, block_index)` yang lolos, sehingga row komponen dari block template ikut terbuang meskipun sempat terbaca.

Setiap worksheet menghasilkan satu row availability, termasuk yang gagal. Exception per sheet ditangkap agar satu sheet rusak tidak menggagalkan seluruh tahun.

In [79]:
def drop_template_blocks(events, components):
    if events.empty:
        return events, components, 0

    template_count = int(events["is_template"].sum())

    kept = events[~events["is_template"]].reset_index(drop=True)

    if components.empty or kept.empty:
        return kept, components.iloc[0:0], template_count

    valid_blocks = set(
        zip(kept["source_sheet"], kept["block_index"])
    )

    mask = [
        (sheet, block) in valid_blocks
        for sheet, block in zip(
            components["source_sheet"],
            components["block_index"]
        )
    ]

    return kept, components[mask].reset_index(drop=True), template_count


def parse_workbook(
    excel_path,
    sheet_names=None,
    block_width=BLOCK_WIDTH,
    verbose=True
):
    xls = pd.ExcelFile(excel_path, engine="openpyxl")
    source_file = Path(excel_path).name

    if sheet_names is None:
        sheet_names = xls.sheet_names

    all_events = []
    all_components = []
    availability = []

    for sheet_index, sheet_name in enumerate(xls.sheet_names, start=1):
        if sheet_name not in sheet_names:
            continue

        record = {
            "source_file": source_file,
            "source_year": extract_year(source_file),
            "sheet_index": sheet_index,
            "source_sheet": sheet_name,
            "block_count": 0,
            "template_block_count": 0,
            "event_count": 0,
            "component_count": 0,
            "status": "OK",
            "message": None,
        }

        try:
            events, components = parse_sheet(
                xls,
                source_file,
                sheet_name,
                sheet_index,
                block_width
            )

            record["block_count"] = len(events)

            events, components, template_count = drop_template_blocks(
                events,
                components
            )

            record["template_block_count"] = template_count
            record["event_count"] = len(events)
            record["component_count"] = (
                0 if components.empty else len(components)
            )

            if record["block_count"] == 0:
                record["status"] = "NO_BLOCK"
                record["message"] = "marker NAMA KOMPONEN tidak ditemukan"
            elif events.empty:
                record["status"] = "TEMPLATE_ONLY"
                record["message"] = "seluruh block masih form kosong"

            if not events.empty:
                all_events.append(events)

            if not components.empty:
                all_components.append(components)

        except Exception as exc:
            record["status"] = "ERROR"
            record["message"] = f"{type(exc).__name__}: {exc}"

            if verbose:
                print(f"  ! {sheet_name}: {record['message']}")

        availability.append(record)

    events_df = (
        pd.concat(all_events, ignore_index=True)
        if all_events
        else pd.DataFrame()
    )

    components_df = (
        pd.concat(all_components, ignore_index=True)
        if all_components
        else pd.DataFrame()
    )

    return events_df, components_df, pd.DataFrame(availability)

## 24. Parse seluruh tahun 2019-2026

In [80]:
def parse_all_workbooks(
    files=None,
    sheet_names=SHEETS_TO_PROCESS,
    block_width=BLOCK_WIDTH
):
    files = files if files is not None else EXCEL_FILES

    events_parts = []
    components_parts = []
    availability_parts = []

    started = time.time()

    for path in files:
        events, components, availability = parse_workbook(
            path,
            sheet_names,
            block_width
        )

        if not events.empty:
            events_parts.append(events)

        if not components.empty:
            components_parts.append(components)

        availability_parts.append(availability)

        print(
            f"{path.name:48} "
            f"sheet={len(availability):3}  "
            f"event={0 if events.empty else len(events):4}  "
            f"template={int(availability['template_block_count'].sum()):3}  "
            f"komponen={0 if components.empty else len(components):6}  "
            f"({time.time() - started:5.1f}s)"
        )

    events_df = pd.concat(events_parts, ignore_index=True)
    components_df = pd.concat(components_parts, ignore_index=True)
    availability_df = pd.concat(availability_parts, ignore_index=True)

    print()
    print("Block terbaca    :", int(availability_df["block_count"].sum()))
    print("Block template   :", int(availability_df["template_block_count"].sum()), "(dibuang)")
    print("Event tersimpan  :", len(events_df))
    print("Komponen         :", len(components_df))
    print("Durasi           :", round(time.time() - started, 1), "detik")

    return events_df, components_df, availability_df


events_df, components_df, availability_df = parse_all_workbooks()

Daftar Nomor Equipment Lokomotif 2019.xlsx       sheet= 36  event=  72  template=  0  komponen=  4388  (  2.9s)
Daftar Nomor Equipment Lokomotif 2020.xlsx       sheet= 34  event=  67  template=  1  komponen=  4003  (  5.6s)
Daftar Nomor Equipment Lokomotif 2021.xlsx       sheet= 41  event=  78  template=  2  komponen=  4777  (  8.2s)
Daftar Nomor Equipment Lokomotif 2022.xlsx       sheet= 55  event= 104  template=  6  komponen=  6421  ( 11.8s)
Daftar Nomor Equipment Lokomotif 2023.xlsx       sheet= 56  event= 107  template=  5  komponen=  6617  ( 15.8s)
Daftar Nomor Equipment Lokomotif 2024.xlsx       sheet= 71  event= 125  template= 17  komponen=  7885  ( 20.5s)
Daftar Nomor Equipment Lokomotif 2025.xlsx       sheet= 69  event= 122  template= 16  komponen=  7636  ( 25.1s)
Daftar Nomor Equipment Lokomotif 2026.xlsx       sheet= 54  event=  96  template= 12  komponen=  7324  ( 29.5s)

Block terbaca    : 830
Block template   : 59 (dibuang)
Event tersimpan  : 771
Komponen         : 49051


# PART E - Penyesuaian tanggal

Kolom `MASUK` dan `KELUAR` tidak selalu ada di file sumber. Form 2019 sama sekali tidak punya kedua kolom itu, dan sebagian block 2020 mengosongkannya.

Untuk row seperti itu tanggal diturunkan dari `PROGRAM BULAN`:

```text
masuk  = tanggal 1 bulan tersebut
keluar = tanggal terakhir bulan tersebut
```

Tahunnya diambil dari tahun file, kecuali `PROGRAM BULAN` menyebut tahun sendiri.

## 25. Menerjemahkan nama bulan

In [81]:
BULAN_INDONESIA = {
    "JANUARI": 1,
    "FEBRUARI": 2,
    "MARET": 3,
    "APRIL": 4,
    "MEI": 5,
    "JUNI": 6,
    "JULI": 7,
    "AGUSTUS": 8,
    "SEPTEMBER": 9,
    "OKTOBER": 10,
    "NOVEMBER": 11,
    "DESEMBER": 12,
}

BULAN_SINGKAT = {
    "JAN": 1,
    "FEB": 2,
    "PEB": 2,
    "MAR": 3,
    "APR": 4,
    "MEI": 5,
    "JUN": 6,
    "JUL": 7,
    "AGU": 8,
    "AGS": 8,
    "AGT": 8,
    "SEP": 9,
    "SEPT": 9,
    "OKT": 10,
    "NOP": 11,
    "NOV": 11,
    "DES": 12,
}


def parse_program_bulan(value, fallback_year):
    """PROGRAM BULAN -> (tanggal 1, tanggal akhir bulan)."""
    text = normalize_label(value)

    if text == "":
        return None, None

    month = None

    for token in text.split():
        if token in BULAN_INDONESIA:
            month = BULAN_INDONESIA[token]
            break

        if token in BULAN_SINGKAT:
            month = BULAN_SINGKAT[token]
            break

    if month is None:
        return None, None

    year = fallback_year
    match = re.search(r"(20\d{2})", text)

    if match:
        year = int(match.group(1))

    if year is None:
        return None, None

    last_day = calendar.monthrange(year, month)[1]

    return date(year, month, 1), date(year, month, last_day)


for sample in ["JANUARI", "Februari", "MARET 2020", "OK", "Rekabeling"]:
    print(f"{sample:14} -> {parse_program_bulan(sample, 2019)}")

JANUARI        -> (datetime.date(2019, 1, 1), datetime.date(2019, 1, 31))
Februari       -> (datetime.date(2019, 2, 1), datetime.date(2019, 2, 28))
MARET 2020     -> (datetime.date(2020, 3, 1), datetime.date(2020, 3, 31))
OK             -> (None, None)
Rekabeling     -> (None, None)


## 26. Mengisi tanggal yang kosong

Tanggal hasil turunan ditandai lewat `masuk_source` dan `keluar_source`, supaya tanggal perkiraan tidak tercampur dengan tanggal yang benar-benar tercatat di Excel.

```text
excel          -> tanggal tertulis di file
program_bulan  -> diturunkan dari PROGRAM BULAN
NULL           -> tidak ada keduanya
```

In [82]:
def fill_dates_from_program(events):
    df = events.copy()

    masuk_values = []
    keluar_values = []
    masuk_sources = []
    keluar_sources = []

    for _, row in df.iterrows():
        masuk = row["masuk"]
        keluar = row["keluar"]

        masuk_source = "excel" if pd.notna(masuk) else None
        keluar_source = "excel" if pd.notna(keluar) else None

        if masuk_source is None or keluar_source is None:
            awal, akhir = parse_program_bulan(
                row["program_bulan"],
                row["source_year"]
            )

            if masuk_source is None and awal is not None:
                masuk = awal
                masuk_source = "program_bulan"

            if keluar_source is None and akhir is not None:
                keluar = akhir
                keluar_source = "program_bulan"

        masuk_values.append(masuk if pd.notna(masuk) else None)
        keluar_values.append(keluar if pd.notna(keluar) else None)
        masuk_sources.append(masuk_source)
        keluar_sources.append(keluar_source)

    df["masuk"] = masuk_values
    df["keluar"] = keluar_values
    df["masuk_source"] = masuk_sources
    df["keluar_source"] = keluar_sources

    return df


before_masuk = int(events_df["masuk"].isna().sum())
before_keluar = int(events_df["keluar"].isna().sum())

events_df = fill_dates_from_program(events_df)

print("masuk kosong  :", before_masuk, "->", int(events_df["masuk"].isna().sum()))
print("keluar kosong :", before_keluar, "->", int(events_df["keluar"].isna().sum()))
print()

display(
    events_df
    .groupby(["source_year", "masuk_source"])
    .size()
    .unstack(fill_value=0)
)

masuk kosong  : 85 -> 3
keluar kosong : 105 -> 23



masuk_source,excel,program_bulan
source_year,,
2019,0,70
2020,55,12
2021,78,0
2022,103,0
2023,107,0
2024,125,0
2025,122,0
2026,96,0


## 27. Tahun maintenance

Diambil dari tahun `masuk`, mundur ke `keluar`, lalu ke tahun file kalau keduanya kosong.

In [83]:
def maintenance_year(row):
    for value in (row["masuk"], row["keluar"]):
        if pd.notna(value):
            return value.year

    return row["source_year"]


events_df["tahun_maintenance"] = events_df.apply(
    maintenance_year,
    axis=1
)

beda = events_df[
    events_df["tahun_maintenance"] != events_df["source_year"]
]

print("Event dengan tahun maintenance beda dari tahun file:", len(beda))

if len(beda):
    display(
        beda[[
            "source_file",
            "source_sheet",
            "block_index",
            "lokomotif_no",
            "program_bulan",
            "masuk",
            "keluar",
            "source_year",
            "tahun_maintenance",
        ]]
    )

Event dengan tahun maintenance beda dari tahun file: 0


## 28. Melengkapi tabel komponen

Tabel komponen ikut membawa identitas lokomotif dan tanggal perawatannya, sehingga bisa dianalisis tanpa harus join ke tabel event lebih dulu.

`how="inner"` sekaligus menjadi jaring pengaman terakhir: komponen yang blocknya tidak ada di tabel event - misalnya block template yang lolos - otomatis terbuang.

In [84]:
EVENT_KEYS = ["source_file", "source_sheet", "block_index"]

COMPONENT_INHERITED = [
    "lokomotif_no",
    "lokomotif_key",
    "masuk",
    "keluar",
    "tahun_maintenance",
]

before_components = len(components_df)

components_df = components_df.merge(
    events_df[EVENT_KEYS + COMPONENT_INHERITED],
    on=EVENT_KEYS,
    how="inner"
)

components_df = components_df[
    [
        "source_file",
        "source_year",
        "source_sheet",
        "block_index",
    ]
    + COMPONENT_INHERITED
    + COMPONENT_COLUMNS
].reset_index(drop=True)

print("Komponen:", before_components, "->", len(components_df))

display(components_df.head(15))

Komponen: 49051 -> 49051


,source_file,source_year,source_sheet,block_index,lokomotif_no,lokomotif_key,masuk,keluar,tahun_maintenance,component_no,component_name,asal_kode_cetak,asal_no_manuf,pengganti_kode_cetak,pengganti_no_manuf,keterangan
0,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,1.0,DIESEL ENGINE,MD-1150780,299235,None,None,None
1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,2.0,CYLINDER ASSY,CA-1150784,CG 92100594,None,None,None
2,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,2.0,CYLINDER ASSY,CA-1150792,-,None,None,None
3,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,2.0,CYLINDER ASSY,CA-1150800,-,None,None,None
4,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,2.0,CYLINDER ASSY,CA-1150808,-,None,None,None
5,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,2.0,CYLINDER ASSY,CA-1150816,-,None,None,None
6,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,2.0,CYLINDER ASSY,CA-1150824,-,None,None,None
7,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,2.0,CYLINDER ASSY,CA-1150831,-,None,None,None
8,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,2.0,CYLINDER ASSY,CA-1150839,-,None,None,None
9,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2039808,2019-01-01,2019-01-31,2019,3.0,OIL PUMP,OP-1150782,E 72360,None,None,None


## 29. Komponen tanpa nomor lokomotif

Pemeriksaan penutup untuk aturan template: setelah merge, tidak boleh ada row komponen tanpa identitas lokomotif.

In [85]:
yatim = components_df[components_df["lokomotif_no"].isna()]

print("Komponen tanpa nomor lokomotif:", len(yatim))

if len(yatim):
    display(yatim.head(20))

Komponen tanpa nomor lokomotif: 0


## 30. Pemasangan baru

Row tanpa komponen asal tetapi punya komponen pengganti berarti pemasangan baru, bukan penggantian. Row seperti ini sengaja dipertahankan.

In [86]:
pemasangan_baru = components_df[
    components_df["asal_kode_cetak"].isna()
    & components_df["asal_no_manuf"].isna()
    & (
        components_df["pengganti_kode_cetak"].notna()
        | components_df["pengganti_no_manuf"].notna()
    )
]

print("Row pemasangan baru:", len(pemasangan_baru))
print()

display(
    pemasangan_baru
    .groupby("tahun_maintenance")
    .size()
    .rename("jumlah")
    .reset_index()
)

display(
    pemasangan_baru[[
        "source_file",
        "source_sheet",
        "block_index",
        "lokomotif_no",
        "component_no",
        "component_name",
        "pengganti_kode_cetak",
        "pengganti_no_manuf",
        "keterangan",
    ]].head(15)
)

Row pemasangan baru: 2921



,tahun_maintenance,jumlah
0,2019,1
1,2020,15
2,2021,79
3,2022,508
4,2023,779
5,2024,383
6,2025,560
7,2026,596


,source_file,source_sheet,block_index,lokomotif_no,component_no,component_name,pengganti_kode_cetak,pengganti_no_manuf,keterangan
1297,Daftar Nomor Equipment Lokomotif 2019.xlsx,Sheet1 (12),2,CC 201 83 21,4.0,WATER PUMP,WP-1149534,GC 92010002,Ex CC 201 78 01
5472,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (25),1,CC 203 98 13,26.0,WHEEL SETS (PERANGKAT RODA),None,41508,Baru
5473,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (25),1,CC 203 98 13,26.0,WHEEL SETS (PERANGKAT RODA),None,42626,Baru
5474,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (25),1,CC 203 98 13,26.0,WHEEL SETS (PERANGKAT RODA),None,43054,Baru
5475,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (25),1,CC 203 98 13,26.0,WHEEL SETS (PERANGKAT RODA),None,43050,Baru
5476,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (25),1,CC 203 98 13,26.0,WHEEL SETS (PERANGKAT RODA),None,42606,Baru
5477,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (25),1,CC 203 98 13,26.0,WHEEL SETS (PERANGKAT RODA),None,13076,Baru
5500,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (25),2,CC 203 98 01,7.0,INJECTION NOZZLE,IN-1421976,0101031090,None
6119,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (20),2,CC 204 03 06,9.0,OVER SPEED GOVERNOR,None,16444677,None
6220,Daftar Nomor Equipment Lokomotif 2020.xlsx,Sheet (19),2,CC 201 83 27 65,7.0,INJECTION NOZZLE,IN-1155858,0101030030,None


# PART F - Data availability

Empat sudut pandang:

1. availability per tahun,
2. worksheet yang tidak menghasilkan data,
3. kelengkapan metadata per tahun,
4. cakupan lokomotif lintas tahun.

## 31. Availability per tahun

In [87]:
sheet_summary = (
    availability_df
    .groupby("source_year")
    .agg(
        sheets=("source_sheet", "size"),
        sheets_ok=("status", lambda s: int((s == "OK").sum())),
        sheets_bermasalah=("status", lambda s: int((s != "OK").sum())),
        blocks=("block_count", "sum"),
        blocks_template=("template_block_count", "sum"),
    )
)

block_summary = (
    events_df
    .groupby("source_year")
    .agg(
        events=("block_index", "size"),
        lokomotif_unik=("lokomotif_key", "nunique"),
        komponen=("component_count", "sum"),
    )
)

availability_by_year = sheet_summary.join(block_summary)

availability_by_year["komponen_per_lok"] = (
    availability_by_year["komponen"]
    / availability_by_year["events"]
).round(1)

display(availability_by_year)

,sheets,sheets_ok,sheets_bermasalah,blocks,blocks_template,events,lokomotif_unik,komponen,komponen_per_lok
source_year,,,,,,,,,
2019,36,36,0,72,0,72,72,4388,60.9
2020,34,34,0,68,1,67,67,4003,59.7
2021,41,39,2,80,2,78,78,4777,61.2
2022,55,52,3,110,6,104,104,6421,61.7
2023,56,54,2,112,5,107,107,6617,61.8
2024,71,63,8,142,17,125,124,7885,63.1
2025,69,62,7,138,16,122,122,7636,62.6
2026,54,48,6,108,12,96,96,7324,76.3


## 32. Worksheet yang tidak menghasilkan data

`NO_BLOCK` berarti marker `NAMA KOMPONEN` tidak ditemukan sama sekali - biasanya sheet kosong atau sheet catatan. `TEMPLATE_ONLY` berarti seluruh block pada sheet itu masih form kosong.

In [88]:
problem_sheets = availability_df[
    availability_df["status"] != "OK"
]

print("Worksheet tanpa data:", len(problem_sheets))
print()

display(
    problem_sheets[[
        "source_year",
        "source_file",
        "sheet_index",
        "source_sheet",
        "block_count",
        "template_block_count",
        "status",
        "message",
    ]]
)

print()
print("Sheet yang sebagian blocknya template:")

display(
    availability_df[
        (availability_df["template_block_count"] > 0)
        & (availability_df["event_count"] > 0)
    ][[
        "source_year",
        "source_sheet",
        "block_count",
        "template_block_count",
        "event_count",
    ]]
)

Worksheet tanpa data: 28



,source_year,source_file,sheet_index,source_sheet,block_count,template_block_count,status,message
70,2021,Daftar Nomor Equipment Lokomotif 2021.xlsx,1,sheet (2),2,2,TEMPLATE_ONLY,seluruh block masih form kosong
110,2021,Daftar Nomor Equipment Lokomotif 2021.xlsx,41,Sheet1,0,0,NO_BLOCK,marker NAMA KOMPONEN tidak ditemukan
163,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,53,sheet (53),2,2,TEMPLATE_ONLY,seluruh block masih form kosong
164,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,54,sheet (54),2,2,TEMPLATE_ONLY,seluruh block masih form kosong
165,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,55,sheet (55),2,2,TEMPLATE_ONLY,seluruh block masih form kosong
166,2023,Daftar Nomor Equipment Lokomotif 2023.xlsx,1,Sheet2 (16),2,2,TEMPLATE_ONLY,seluruh block masih form kosong
221,2023,Daftar Nomor Equipment Lokomotif 2023.xlsx,56,Sheet2 (51),2,2,TEMPLATE_ONLY,seluruh block masih form kosong
285,2024,Daftar Nomor Equipment Lokomotif 2024.xlsx,64,Sheet (48),2,2,TEMPLATE_ONLY,seluruh block masih form kosong
286,2024,Daftar Nomor Equipment Lokomotif 2024.xlsx,65,Sheet (49),2,2,TEMPLATE_ONLY,seluruh block masih form kosong
287,2024,Daftar Nomor Equipment Lokomotif 2024.xlsx,66,Sheet (50),2,2,TEMPLATE_ONLY,seluruh block masih form kosong



Sheet yang sebagian blocknya template:


,source_year,source_sheet,block_count,template_block_count,event_count
36,2020,Sheet (34),2,1,1
220,2023,1369,2,1,1
284,2024,1339,2,1,1
323,2025,8916,2,1,1
354,2025,1394,2,1,1


## 33. Kelengkapan metadata per tahun

In [89]:
METADATA_FIELDS = [
    "no_seri_lokomotif",
    "dipo_induk",
    "jenis_perawatan",
    "program_bulan",
    "masuk",
    "keluar",
]

completeness = (
    events_df
    .groupby("source_year")[METADATA_FIELDS]
    .apply(lambda g: (g.notna().mean() * 100).round(1))
)

completeness.insert(
    0,
    "events",
    events_df.groupby("source_year").size()
)

print("Persentase field terisi, setelah tanggal dilengkapi dari PROGRAM BULAN\n")

display(completeness)

Persentase field terisi, setelah tanggal dilengkapi dari PROGRAM BULAN



,events,no_seri_lokomotif,dipo_induk,jenis_perawatan,program_bulan,masuk,keluar
source_year,,,,,,,
2019,72,100.0,97.2,100.0,97.2,97.2,97.2
2020,67,100.0,100.0,100.0,41.8,100.0,100.0
2021,78,100.0,100.0,100.0,15.4,100.0,98.7
2022,104,100.0,100.0,100.0,13.5,99.0,100.0
2023,107,100.0,100.0,100.0,0.9,100.0,100.0
2024,125,100.0,100.0,100.0,0.0,100.0,100.0
2025,122,100.0,100.0,100.0,0.0,100.0,100.0
2026,96,100.0,100.0,100.0,0.0,100.0,79.2


## 34. Cakupan lokomotif lintas tahun

In [90]:
coverage = (
    events_df
    .dropna(subset=["lokomotif_key"])
    .pivot_table(
        index="lokomotif_no",
        columns="tahun_maintenance",
        values="block_index",
        aggfunc="size",
        fill_value=0
    )
)

coverage["total_masuk"] = coverage.sum(axis=1)
coverage["tahun_hadir"] = (coverage.iloc[:, :-1] > 0).sum(axis=1)

print("Lokomotif unik sepanjang 2019-2026:", len(coverage))
print()

display(
    coverage
    .sort_values("total_masuk", ascending=False)
    .head(25)
)

Lokomotif unik sepanjang 2019-2026: 279



tahun_maintenance,2019,2020,2021,2022,2023,2024,2025,2026,total_masuk,tahun_hadir
lokomotif_no,,,,,,,,,,
CC 203 98 03,1,0,1,0,1,0,1,0,4,4
CC 203 95 09,1,0,0,1,0,1,0,1,4,4
CC 201 77 21,1,0,0,1,0,1,0,1,4,4
CC 203 95 10,1,0,0,1,0,1,0,1,4,4
CC 201 92 09,1,0,0,1,0,1,0,1,4,4
CC 201 78 01,1,0,0,1,0,1,0,1,4,4
CC 203 95 11,1,0,1,0,1,0,1,0,4,4
CC 201 83 33,1,0,0,1,0,1,0,1,4,4
CC 201 78 05,1,0,0,1,0,1,0,1,4,4


## 35. Distribusi kehadiran lokomotif

In [91]:
print("Jumlah lokomotif menurut banyaknya tahun kemunculan\n")

display(
    coverage["tahun_hadir"]
    .value_counts()
    .sort_index()
    .rename_axis("jumlah_tahun")
    .reset_index(name="jumlah_lokomotif")
)

Jumlah lokomotif menurut banyaknya tahun kemunculan



,jumlah_tahun,jumlah_lokomotif
0,1,36
1,2,41
2,3,156
3,4,46


# PART G - Validasi

Selain kelengkapan, dicek juga nilai yang keluar dari domain yang wajar.

## 36. Komponen tanpa nama

In [92]:
invalid_components = components_df[
    components_df["component_name"].isna()
]

print("Komponen tanpa nama:", len(invalid_components))

if len(invalid_components):
    display(
        invalid_components[[
            "source_file",
            "source_sheet",
            "block_index",
            "component_no",
            "asal_kode_cetak",
        ]].head(20)
    )

Komponen tanpa nama: 0


## 37. Nilai dipo dan jenis perawatan di luar domain

Beberapa row sumber tertukar isinya, misalnya `DIPO INDUK = P.48` sementara `JENIS PERAWATAN = SMC`. Ini kesalahan pengetikan di file Excel, bukan kesalahan parsing, karena itu hanya dilaporkan - tidak diperbaiki otomatis. Kolom referensi sudah cukup untuk membuka sheet dan blocknya.

In [93]:
KNOWN_DIPO = {
    "SDT", "CPN", "YK", "PWT", "BD", "SMC", "CN",
    "JR", "MN", "JNG", "THB", "SMG", "BYYK", "TNK",
}

suspect = events_df[
    events_df["dipo_induk"].notna()
    & ~events_df["dipo_induk"].str.upper().str.strip().isin(KNOWN_DIPO)
]

print("Nilai dipo_induk di luar daftar:", len(suspect))

display(
    suspect[[
        "source_year",
        "source_file",
        "source_sheet",
        "block_index",
        "lokomotif_no",
        "dipo_induk",
        "jenis_perawatan",
    ]]
)

Nilai dipo_induk di luar daftar: 1


,source_year,source_file,source_sheet,block_index,lokomotif_no,dipo_induk,jenis_perawatan
502,2024,Daftar Nomor Equipment Lokomotif 2024.xlsx,1366 9501,1,CC 206 13 66,P.48,SMC


## 38. Duplicate check

Nama seperti `INJECTION NOZZLE` wajar muncul berkali-kali dalam satu block. Yang diperiksa di sini adalah row yang identik seluruh isinya dalam block yang sama.

In [94]:
DUPLICATE_COLS = [
    "source_file",
    "source_sheet",
    "block_index",
] + COMPONENT_COLUMNS

duplicates = components_df[
    components_df.duplicated(subset=DUPLICATE_COLS, keep=False)
]

print("Row duplikat penuh:", len(duplicates))

if len(duplicates):
    display(
        duplicates
        .sort_values(DUPLICATE_COLS)
        [[
            "source_year",
            "source_sheet",
            "block_index",
            "lokomotif_no",
        ] + COMPONENT_COLUMNS]
        .head(30)
    )

Row duplikat penuh: 242


,source_year,source_sheet,block_index,lokomotif_no,component_no,component_name,asal_kode_cetak,asal_no_manuf,pengganti_kode_cetak,pengganti_no_manuf,keterangan
2081,2019,Sheet1 (19),1,CC 206 15 07,2.0,CYLINDER ASSY,None,-,None,None,None
2082,2019,Sheet1 (19),1,CC 206 15 07,2.0,CYLINDER ASSY,None,-,None,None,None
2083,2019,Sheet1 (19),1,CC 206 15 07,2.0,CYLINDER ASSY,None,-,None,None,None
2084,2019,Sheet1 (19),1,CC 206 15 07,2.0,CYLINDER ASSY,None,-,None,None,None
2085,2019,Sheet1 (19),1,CC 206 15 07,2.0,CYLINDER ASSY,None,-,None,None,None
2086,2019,Sheet1 (19),1,CC 206 15 07,2.0,CYLINDER ASSY,None,-,None,None,None
2087,2019,Sheet1 (19),1,CC 206 15 07,2.0,CYLINDER ASSY,None,-,None,None,None
2088,2019,Sheet1 (19),1,CC 206 15 07,2.0,CYLINDER ASSY,None,-,None,None,None
2093,2019,Sheet1 (19),1,CC 206 15 07,7.0,INJECTION NOZZLE,None,-,None,None,None
2094,2019,Sheet1 (19),1,CC 206 15 07,7.0,INJECTION NOZZLE,None,-,None,None,None


# PART H - SQLite schema v3

Empat tabel:

```text
maintenance_events      parent, satu row per block lokomotif yang terpakai
equipment_components    child, satu row per equipment
sheet_availability      hasil scan setiap worksheet
ingestion_history       log setiap kali ingestion dijalankan
```

Referensi asal yang ada di kedua tabel data: `source_file`, `source_year`, `source_sheet`, `block_index`. Tabel komponen menambahkan identitas perawatannya sendiri - `lokomotif_no`, `lokomotif_key`, `masuk`, `keluar`, `tahun_maintenance`.

## 39. Connection

`isolation_level=None` membuat koneksi berjalan autocommit, sehingga blok transaksi bisa dikendalikan eksplisit lewat `BEGIN` / `COMMIT` pada tahap ingestion.

In [95]:
from contextlib import closing


def get_connection():
    conn = sqlite3.connect(DB_FILE, isolation_level=None)
    conn.execute("PRAGMA foreign_keys = ON;")
    conn.row_factory = sqlite3.Row

    return conn

## 40. DDL

In [96]:
CREATE_EVENTS_SQL = """
CREATE TABLE IF NOT EXISTS maintenance_events (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    source_file TEXT NOT NULL,
    source_year INTEGER,
    source_sheet TEXT NOT NULL,
    block_index INTEGER NOT NULL,

    no_seri_lokomotif TEXT,
    lokomotif_no TEXT,
    lokomotif_key TEXT,

    dipo_induk TEXT,
    jenis_perawatan TEXT,
    program_bulan TEXT,
    ganti_di_dipo TEXT,

    masuk TEXT,
    keluar TEXT,
    masuk_source TEXT,
    keluar_source TEXT,
    tahun_maintenance INTEGER,

    component_count INTEGER DEFAULT 0,

    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,

    UNIQUE (source_file, source_sheet, block_index)
);
"""

CREATE_COMPONENTS_SQL = """
CREATE TABLE IF NOT EXISTS equipment_components (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    maintenance_event_id INTEGER NOT NULL,

    source_file TEXT NOT NULL,
    source_year INTEGER,
    source_sheet TEXT NOT NULL,
    block_index INTEGER NOT NULL,

    lokomotif_no TEXT,
    lokomotif_key TEXT,
    masuk TEXT,
    keluar TEXT,
    tahun_maintenance INTEGER,

    component_no TEXT,
    component_name TEXT,

    asal_kode_cetak TEXT,
    asal_no_manuf TEXT,

    pengganti_kode_cetak TEXT,
    pengganti_no_manuf TEXT,

    keterangan TEXT,

    FOREIGN KEY (maintenance_event_id)
        REFERENCES maintenance_events(id)
        ON DELETE CASCADE
);
"""

CREATE_AVAILABILITY_SQL = """
CREATE TABLE IF NOT EXISTS sheet_availability (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    source_file TEXT NOT NULL,
    source_year INTEGER,
    sheet_index INTEGER,
    source_sheet TEXT NOT NULL,

    block_count INTEGER DEFAULT 0,
    template_block_count INTEGER DEFAULT 0,
    event_count INTEGER DEFAULT 0,
    component_count INTEGER DEFAULT 0,

    status TEXT NOT NULL,
    message TEXT,

    scanned_at TEXT NOT NULL,

    UNIQUE (source_file, source_sheet)
);
"""

CREATE_LOG_SQL = """
CREATE TABLE IF NOT EXISTS ingestion_history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    source_file TEXT NOT NULL,
    source_year INTEGER,
    schema_version INTEGER,

    started_at TEXT NOT NULL,
    finished_at TEXT,

    sheet_count INTEGER DEFAULT 0,
    event_count INTEGER DEFAULT 0,
    component_count INTEGER DEFAULT 0,

    status TEXT NOT NULL,
    error_message TEXT
);
"""

CREATE_INDEX_SQL = [
    "CREATE INDEX IF NOT EXISTS idx_events_loco ON maintenance_events(lokomotif_key);",
    "CREATE INDEX IF NOT EXISTS idx_events_tahun ON maintenance_events(tahun_maintenance);",
    "CREATE INDEX IF NOT EXISTS idx_events_source ON maintenance_events(source_file, source_sheet);",
    "CREATE INDEX IF NOT EXISTS idx_components_event ON equipment_components(maintenance_event_id);",
    "CREATE INDEX IF NOT EXISTS idx_components_loco ON equipment_components(lokomotif_key);",
    "CREATE INDEX IF NOT EXISTS idx_components_tahun ON equipment_components(tahun_maintenance);",
    "CREATE INDEX IF NOT EXISTS idx_components_name ON equipment_components(component_name);",
    "CREATE INDEX IF NOT EXISTS idx_components_asal ON equipment_components(asal_kode_cetak);",
]

ALL_TABLES = [
    "equipment_components",
    "maintenance_events",
    "sheet_availability",
    "ingestion_history",
]

## 41. Migrasi schema

Karena seluruh isi database berasal dari Excel, cara paling bersih saat struktur berubah adalah membangun ulang - dengan file lama di-backup lebih dulu.

Versi schema disimpan pada `PRAGMA user_version`.

In [97]:
def ensure_schema(rebuild_if_outdated=REBUILD_DB_IF_SCHEMA_OUTDATED):
    with closing(get_connection()) as conn:
        current_version = conn.execute("PRAGMA user_version;").fetchone()[0]

        existing_tables = {
            row["name"]
            for row in conn.execute(
                "SELECT name FROM sqlite_master WHERE type = 'table';"
            )
        }

        outdated = bool(existing_tables) and current_version < SCHEMA_VERSION

    if outdated:
        if not rebuild_if_outdated:
            raise RuntimeError(
                f"Database schema v{current_version}, "
                f"dibutuhkan v{SCHEMA_VERSION}. "
                "Set REBUILD_DB_IF_SCHEMA_OUTDATED = True untuk rebuild."
            )

        backup = DB_FILE.with_name(
            f"{DB_FILE.stem}_backup_v{current_version}.db"
        )

        shutil.copy2(DB_FILE, backup)

        print(f"Schema lama v{current_version} terdeteksi.")
        print(f"Backup dibuat  : {backup.name}")
        print("Tabel lama dibangun ulang dari Excel.")

        with closing(get_connection()) as conn:
            for table in ALL_TABLES:
                conn.execute(f"DROP TABLE IF EXISTS {table};")

    with closing(get_connection()) as conn:
        for ddl in (
            CREATE_EVENTS_SQL,
            CREATE_COMPONENTS_SQL,
            CREATE_AVAILABILITY_SQL,
            CREATE_LOG_SQL,
        ):
            conn.execute(ddl)

        for statement in CREATE_INDEX_SQL:
            conn.execute(statement)

        conn.execute(f"PRAGMA user_version = {SCHEMA_VERSION};")

        tables = [
            row["name"]
            for row in conn.execute(
                "SELECT name FROM sqlite_master "
                "WHERE type = 'table' ORDER BY name;"
            )
        ]

    print("Schema version :", SCHEMA_VERSION)
    print("Tabel          :", tables)


ensure_schema()

Schema version : 3
Tabel          : ['equipment_components', 'ingestion_history', 'maintenance_events', 'sheet_availability', 'sqlite_sequence']


# PART I - Transactional ingestion

Strategi per file:

```text
BEGIN
    UPSERT maintenance event
    DELETE komponen lama milik event tersebut
    INSERT komponen baru
    UPSERT sheet_availability
COMMIT
```

Ingestion bersifat idempoten: menjalankan ulang notebook tidak menggandakan data, karena event dikunci pada `(source_file, source_sheet, block_index)` dan komponennya selalu ditulis ulang.

## 42. Konversi nilai ke SQLite

In [98]:
def sqlite_value(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    if hasattr(value, "isoformat"):
        try:
            return value.isoformat()
        except Exception:
            pass

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, (float, np.floating)):
        number = float(value)

        # Kolom NO. terbaca pandas sebagai float, sehingga nomor komponen
        # 1 akan tersimpan "1.0" kalau tidak dikembalikan ke int lebih dulu.
        if number.is_integer():
            return int(number)

        return number

    if isinstance(value, int):
        return value

    return str(value)


def sqlite_int(value):
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    return int(value)

## 43. Statement upsert dan insert

In [99]:
EVENT_FIELDS = [
    "source_file",
    "source_year",
    "source_sheet",
    "block_index",
    "no_seri_lokomotif",
    "lokomotif_no",
    "lokomotif_key",
    "dipo_induk",
    "jenis_perawatan",
    "program_bulan",
    "ganti_di_dipo",
    "masuk",
    "keluar",
    "masuk_source",
    "keluar_source",
    "tahun_maintenance",
    "component_count",
]

EVENT_UPDATE_FIELDS = [
    f
    for f in EVENT_FIELDS
    if f not in ("source_file", "source_sheet", "block_index")
]

UPSERT_EVENT_SQL = f"""
INSERT INTO maintenance_events (
    {", ".join(EVENT_FIELDS)},
    created_at,
    updated_at
)
VALUES ({", ".join(["?"] * (len(EVENT_FIELDS) + 2))})

ON CONFLICT (source_file, source_sheet, block_index)
DO UPDATE SET
    {", ".join(f"{f} = excluded.{f}" for f in EVENT_UPDATE_FIELDS)},
    updated_at = excluded.updated_at;
"""

COMPONENT_FIELDS = [
    "maintenance_event_id",
    "source_file",
    "source_year",
    "source_sheet",
    "block_index",
    "lokomotif_no",
    "lokomotif_key",
    "masuk",
    "keluar",
    "tahun_maintenance",
    "component_no",
    "component_name",
    "asal_kode_cetak",
    "asal_no_manuf",
    "pengganti_kode_cetak",
    "pengganti_no_manuf",
    "keterangan",
]

INSERT_COMPONENT_SQL = f"""
INSERT INTO equipment_components (
    {", ".join(COMPONENT_FIELDS)}
)
VALUES ({", ".join(["?"] * len(COMPONENT_FIELDS))});
"""

UPSERT_AVAILABILITY_SQL = """
INSERT INTO sheet_availability (
    source_file,
    source_year,
    sheet_index,
    source_sheet,
    block_count,
    template_block_count,
    event_count,
    component_count,
    status,
    message,
    scanned_at
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)

ON CONFLICT (source_file, source_sheet)
DO UPDATE SET
    source_year = excluded.source_year,
    sheet_index = excluded.sheet_index,
    block_count = excluded.block_count,
    template_block_count = excluded.template_block_count,
    event_count = excluded.event_count,
    component_count = excluded.component_count,
    status = excluded.status,
    message = excluded.message,
    scanned_at = excluded.scanned_at;
"""

print("Kolom event    :", len(EVENT_FIELDS))
print("Kolom komponen :", len(COMPONENT_FIELDS))

Kolom event    : 17
Kolom komponen : 17


## 44. Upsert parent

In [100]:
INTEGER_EVENT_FIELDS = {
    "source_year",
    "block_index",
    "tahun_maintenance",
    "component_count",
}


def upsert_event(conn, event_row, timestamp):
    values = []

    for field in EVENT_FIELDS:
        value = event_row[field]

        if field in INTEGER_EVENT_FIELDS:
            values.append(sqlite_int(value))
        else:
            values.append(sqlite_value(value))

    conn.execute(
        UPSERT_EVENT_SQL,
        tuple(values) + (timestamp, timestamp)
    )

    row = conn.execute(
        """
        SELECT id
        FROM maintenance_events
        WHERE source_file = ?
          AND source_sheet = ?
          AND block_index = ?
        """,
        (
            event_row["source_file"],
            event_row["source_sheet"],
            int(event_row["block_index"]),
        )
    ).fetchone()

    return row["id"]

## 45. Replace children

In [101]:
INTEGER_COMPONENT_FIELDS = {
    "source_year",
    "block_index",
    "tahun_maintenance",
}


def replace_components(conn, event_id, component_rows):
    conn.execute(
        """
        DELETE FROM equipment_components
        WHERE maintenance_event_id = ?
        """,
        (event_id,)
    )

    if component_rows.empty:
        return 0

    records = []

    for _, row in component_rows.iterrows():
        values = [event_id]

        for field in COMPONENT_FIELDS[1:]:
            value = row[field]

            if field in INTEGER_COMPONENT_FIELDS:
                values.append(sqlite_int(value))
            else:
                values.append(sqlite_value(value))

        records.append(tuple(values))

    conn.executemany(INSERT_COMPONENT_SQL, records)

    return len(records)

## 46. Ingestion satu file

In [102]:
def ingest_source_file(
    source_file,
    events_df,
    components_df,
    availability_df
):
    file_events = events_df[events_df["source_file"] == source_file]

    file_components = (
        components_df[components_df["source_file"] == source_file]
        if not components_df.empty
        else components_df
    )

    file_availability = availability_df[
        availability_df["source_file"] == source_file
    ]

    started_at = datetime.now(timezone.utc).isoformat()

    with closing(get_connection()) as conn:
        cursor = conn.execute(
            """
            INSERT INTO ingestion_history (
                source_file,
                source_year,
                schema_version,
                started_at,
                status
            )
            VALUES (?, ?, ?, ?, ?)
            """,
            (
                source_file,
                extract_year(source_file),
                SCHEMA_VERSION,
                started_at,
                "RUNNING",
            )
        )

        log_id = cursor.lastrowid
        total_components = 0

        try:
            conn.execute("BEGIN")

            for _, event in file_events.iterrows():
                event_id = upsert_event(conn, event, started_at)

                if file_components.empty:
                    matching = file_components
                else:
                    matching = file_components[
                        (file_components["source_sheet"] == event["source_sheet"])
                        & (file_components["block_index"] == event["block_index"])
                    ]

                total_components += replace_components(
                    conn,
                    event_id,
                    matching
                )

            scanned_at = datetime.now(timezone.utc).isoformat()

            for _, sheet in file_availability.iterrows():
                conn.execute(
                    UPSERT_AVAILABILITY_SQL,
                    (
                        sheet["source_file"],
                        sqlite_int(sheet["source_year"]),
                        sqlite_int(sheet["sheet_index"]),
                        sheet["source_sheet"],
                        sqlite_int(sheet["block_count"]),
                        sqlite_int(sheet["template_block_count"]),
                        sqlite_int(sheet["event_count"]),
                        sqlite_int(sheet["component_count"]),
                        sheet["status"],
                        sqlite_value(sheet["message"]),
                        scanned_at,
                    )
                )

            conn.execute("COMMIT")

            conn.execute(
                """
                UPDATE ingestion_history
                SET finished_at = ?,
                    sheet_count = ?,
                    event_count = ?,
                    component_count = ?,
                    status = ?
                WHERE id = ?
                """,
                (
                    datetime.now(timezone.utc).isoformat(),
                    len(file_availability),
                    len(file_events),
                    total_components,
                    "SUCCESS",
                    log_id,
                )
            )

            return {
                "source_file": source_file,
                "status": "SUCCESS",
                "sheets": len(file_availability),
                "events": len(file_events),
                "components": total_components,
            }

        except Exception as exc:
            conn.execute("ROLLBACK")

            conn.execute(
                """
                UPDATE ingestion_history
                SET finished_at = ?,
                    status = ?,
                    error_message = ?
                WHERE id = ?
                """,
                (
                    datetime.now(timezone.utc).isoformat(),
                    "FAILED",
                    str(exc),
                    log_id,
                )
            )

            raise

## 47. Jalankan ingestion untuk seluruh tahun

In [103]:
ingest_results = []

for path in EXCEL_FILES:
    result = ingest_source_file(
        path.name,
        events_df,
        components_df,
        availability_df
    )

    ingest_results.append(result)
    print(
        f"{result['source_file']:48} "
        f"{result['status']:8} "
        f"event={result['events']:4}  "
        f"komponen={result['components']:6}"
    )

ingest_summary = pd.DataFrame(ingest_results)

print()
print("Total event    :", ingest_summary["events"].sum())
print("Total komponen :", ingest_summary["components"].sum())

Daftar Nomor Equipment Lokomotif 2019.xlsx       SUCCESS  event=  72  komponen=  4388
Daftar Nomor Equipment Lokomotif 2020.xlsx       SUCCESS  event=  67  komponen=  4003
Daftar Nomor Equipment Lokomotif 2021.xlsx       SUCCESS  event=  78  komponen=  4777
Daftar Nomor Equipment Lokomotif 2022.xlsx       SUCCESS  event= 104  komponen=  6421
Daftar Nomor Equipment Lokomotif 2023.xlsx       SUCCESS  event= 107  komponen=  6617
Daftar Nomor Equipment Lokomotif 2024.xlsx       SUCCESS  event= 125  komponen=  7885
Daftar Nomor Equipment Lokomotif 2025.xlsx       SUCCESS  event= 122  komponen=  7636
Daftar Nomor Equipment Lokomotif 2026.xlsx       SUCCESS  event=  96  komponen=  7324

Total event    : 771
Total komponen : 49051


# PART J - Query

## 48. Isi tabel maintenance_events

In [104]:
with closing(get_connection()) as conn:
    db_events = pd.read_sql_query(
        """
        SELECT id,
               source_file,
               source_year,
               source_sheet,
               block_index,
               lokomotif_no,
               dipo_induk,
               jenis_perawatan,
               masuk,
               keluar,
               masuk_source,
               tahun_maintenance,
               component_count
        FROM maintenance_events
        ORDER BY source_year, source_sheet, block_index;
        """,
        conn
    )

print("Rows:", len(db_events))

display(db_events.head(20))

Rows: 771


,id,source_file,source_year,source_sheet,block_index,lokomotif_no,dipo_induk,jenis_perawatan,masuk,keluar,masuk_source,tahun_maintenance,component_count
0,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,program_bulan,2019,61
1,2,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,2,CC 204 03 04,YK,SPA / P24,2019-02-01,2019-02-28,program_bulan,2019,62
2,17,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1 (10),1,CC 203 95 11,MN,SPA 2 / P48,2019-03-01,2019-03-31,program_bulan,2019,61
3,18,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1 (10),2,CC 201 77 21,CN,SPA 2 / P48,2019-03-01,2019-03-31,program_bulan,2019,61
4,19,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1 (11),1,CC 201 78 02,YK,SPA 2 / P48,2019-04-01,2019-04-30,program_bulan,2019,61
5,20,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1 (11),2,CC 201 83 33,SMC,SPA 2 / P48,2019-04-01,2019-04-30,program_bulan,2019,61
6,21,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1 (12),1,CC 201 92 20,JNG,SPA 2 / P48,2019-04-01,2019-04-30,program_bulan,2019,61
7,22,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1 (12),2,CC 201 83 21,PWT,SPA 2 / P48,2019-04-01,2019-04-30,program_bulan,2019,62
8,23,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1 (13),1,CC 201 77 12,SDT,SPA 2 / P48,2019-04-01,2019-04-30,program_bulan,2019,61
9,24,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1 (13),2,CC 201 92 18,JNG,SPA 2 / P48,2019-04-01,2019-04-30,program_bulan,2019,61


## 49. Isi tabel equipment_components

Setiap row komponen sudah membawa asal filenya sekaligus lokomotif dan tanggal perawatannya.

In [105]:
with closing(get_connection()) as conn:
    db_components = pd.read_sql_query(
        """
        SELECT id,
               maintenance_event_id,
               source_file,
               source_year,
               source_sheet,
               block_index,
               lokomotif_no,
               masuk,
               keluar,
               tahun_maintenance,
               component_no,
               component_name,
               asal_kode_cetak,
               asal_no_manuf,
               pengganti_kode_cetak,
               pengganti_no_manuf,
               keterangan
        FROM equipment_components
        ORDER BY maintenance_event_id, id
        LIMIT 200;
        """,
        conn
    )

display(db_components.head(30))

,id,maintenance_event_id,source_file,source_year,source_sheet,block_index,lokomotif_no,masuk,keluar,tahun_maintenance,component_no,component_name,asal_kode_cetak,asal_no_manuf,pengganti_kode_cetak,pengganti_no_manuf,keterangan
0,98103,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,1,DIESEL ENGINE,MD-1150780,299235,None,None,None
1,98104,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150784,CG 92100594,None,None,None
2,98105,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150792,-,None,None,None
3,98106,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150800,-,None,None,None
4,98107,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150808,-,None,None,None
5,98108,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150816,-,None,None,None
6,98109,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150824,-,None,None,None
7,98110,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150831,-,None,None,None
8,98111,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150839,-,None,None,None
9,98112,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,2019-01-01,2019-01-31,2019,3,OIL PUMP,OP-1150782,E 72360,None,None,None


## 50. JOIN parent + child

In [106]:
JOIN_QUERY = """
SELECT
    e.id AS maintenance_event_id,
    e.source_file,
    e.source_year,
    e.source_sheet,
    e.block_index,

    e.lokomotif_no,
    e.dipo_induk,
    e.jenis_perawatan,
    e.masuk,
    e.keluar,
    e.tahun_maintenance,

    c.component_no,
    c.component_name,
    c.asal_kode_cetak,
    c.asal_no_manuf,
    c.pengganti_kode_cetak,
    c.pengganti_no_manuf,
    c.keterangan

FROM maintenance_events e
LEFT JOIN equipment_components c
    ON c.maintenance_event_id = e.id

ORDER BY e.tahun_maintenance, e.id, c.id
LIMIT 500;
"""

with closing(get_connection()) as conn:
    report_df = pd.read_sql_query(JOIN_QUERY, conn)

display(report_df.head(30))

,maintenance_event_id,source_file,source_year,source_sheet,block_index,lokomotif_no,dipo_induk,jenis_perawatan,masuk,keluar,tahun_maintenance,component_no,component_name,asal_kode_cetak,asal_no_manuf,pengganti_kode_cetak,pengganti_no_manuf,keterangan
0,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,1,DIESEL ENGINE,MD-1150780,299235,None,None,None
1,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150784,CG 92100594,None,None,None
2,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150792,-,None,None,None
3,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150800,-,None,None,None
4,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150808,-,None,None,None
5,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150816,-,None,None,None
6,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150824,-,None,None,None
7,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150831,-,None,None,None
8,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,2,CYLINDER ASSY,CA-1150839,-,None,None,None
9,1,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,Sheet1,1,CC 203 98 08,JNG,SPA 2 / P48,2019-01-01,2019-01-31,2019,3,OIL PUMP,OP-1150782,E 72360,None,None,None


## 51. Riwayat satu lokomotif lintas tahun

Pencarian memakai `lokomotif_key` (digit saja) supaya perbedaan penulisan antar tahun tidak menghalangi.

In [107]:
LOCOMOTIVE = "CC 201 83 02"

_, locomotive_key = normalize_loco(LOCOMOTIVE)

with closing(get_connection()) as conn:
    riwayat = pd.read_sql_query(
        """
        SELECT tahun_maintenance,
               source_file,
               source_sheet,
               block_index,
               lokomotif_no,
               dipo_induk,
               jenis_perawatan,
               masuk,
               keluar,
               masuk_source,
               component_count
        FROM maintenance_events
        WHERE lokomotif_key = ?
        ORDER BY tahun_maintenance, masuk;
        """,
        conn,
        params=(locomotive_key,)
    )

print(f"{LOCOMOTIVE} -> key {locomotive_key}, {len(riwayat)} kali perawatan")

display(riwayat)

CC 201 83 02 -> key 2018302, 3 kali perawatan


,tahun_maintenance,source_file,source_sheet,block_index,lokomotif_no,dipo_induk,jenis_perawatan,masuk,keluar,masuk_source,component_count
0,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,CC 201 83 02,YK,P72,2022-02-03,2022-03-07,excel,60
1,2024,Daftar Nomor Equipment Lokomotif 2024.xlsx,9209 8302,2,CC 201 83 02,YK,P.24,2024-01-12,2024-02-01,excel,61
2,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,8302 9209,1,CC 201 83 02,YK,P48,2026-01-07,2026-01-26,excel,76


## 52. Detail komponen satu lokomotif

Karena tabel komponen sudah membawa `lokomotif_key` dan `tahun_maintenance`, query ini tidak perlu join sama sekali.

In [108]:
with closing(get_connection()) as conn:
    detail = pd.read_sql_query(
        """
        SELECT tahun_maintenance,
               source_file,
               source_sheet,
               block_index,
               masuk,
               keluar,
               component_no,
               component_name,
               asal_kode_cetak,
               asal_no_manuf,
               pengganti_kode_cetak,
               pengganti_no_manuf,
               keterangan
        FROM equipment_components
        WHERE lokomotif_key = ?
        ORDER BY tahun_maintenance, id;
        """,
        conn,
        params=(locomotive_key,)
    )

print("Rows:", len(detail))

display(detail.head(40))

Rows: 197


,tahun_maintenance,source_file,source_sheet,block_index,masuk,keluar,component_no,component_name,asal_kode_cetak,asal_no_manuf,pengganti_kode_cetak,pengganti_no_manuf,keterangan
0,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,1,DIESEL ENGINE,MD-1151907,292802,None,None,Kembali
1,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,2,CYLINDER ASSY 1R,CA-1159440,S 8301131-MG 13070758,CA-1163500,LG01110907-MG20030564,EX CC 201 04 07
2,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,2,2R,CA-1159436,CG 91110353-MG 12052021,CA-1163488,LG02050779-MG19120717,EX CC 201 04 07
3,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,2,3R,CA-1159444,CG 98011219-MG 13090203,CA-1163508,CG91110349-MG20080660,EX CC 201 04 07
4,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,2,4R,CA-1403719,CG 96050031,CA-1163512,CG95090278-MG20080660,EX CC 201 04 07
5,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,2,1L,CA-1159452,LG 96041071-MG 13091213,CA-1163484,CG96050061-MG20080661,EX CC 201 04 07
6,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,2,2L,CA-1159456,E 7612237-MG 13091452,CA-1163492,C8304082-MG20080676,EX CC 201 04 07
7,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,2,3L,CA-1159448,CG96050076,CA-1163504,E7609479-MG20080637,EX CC 201 04 07
8,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,2,4L,None,C8304021,CA-1163496,CG96081138-MG20021087,EX CC 201 04 07
9,2022,Daftar Nomor Equipment Lokomotif 2022.xlsx,9509 8302,2,2022-02-03,2022-03-07,3,GOVERNOR MD,GM-1159471,10818,None,None,Kembali


## 53. Riwayat satu komponen

Contoh pemakaian `tahun_maintenance` pada tabel komponen: melacak satu nomor equipment melintasi tahun dan lokomotif.

In [109]:
with closing(get_connection()) as conn:
    riwayat_komponen = pd.read_sql_query(
        """
        SELECT tahun_maintenance,
               source_file,
               source_sheet,
               block_index,
               lokomotif_no,
               masuk,
               component_name,
               asal_kode_cetak,
               asal_no_manuf,
               keterangan
        FROM equipment_components
        WHERE component_name LIKE '%TURBO%'
          AND asal_kode_cetak IS NOT NULL
        ORDER BY tahun_maintenance DESC, id
        LIMIT 15;
        """,
        conn
    )

display(riwayat_komponen)

,tahun_maintenance,source_file,source_sheet,block_index,lokomotif_no,masuk,component_name,asal_kode_cetak,asal_no_manuf,keterangan
0,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,8302 9209,1,CC 201 83 02,2026-01-07,TURBO CHARGER,TC-1436619,CG19120087,EX CC 201 04 07
1,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,8302 9209,2,CC 201 92 09,2026-01-08,TURBO CHARGER,TC-1441268,GC13060079,Kembali
2,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,0203 0407,1,CC 203 02 03,2026-01-08,TURBO CHARGER,TC-1458091,GC19120043,Kembali
3,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,0203 0407,2,CC 201 04 07,2026-01-09,TURBO CHARGER,TC-9005308,GC22010031,EX CC 201 83 02
4,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,1323 1303,1,CC 206 13 23,2026-01-09,TURBO CHARGER,TC-1149627,GC14070053,CU
5,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,1323 1303,2,CC 206 13 03,2026-01-09,TURBO CHARGER,TC-1447065,GC12090113,None
6,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,1350 7712,1,CC 206 13 50,2026-01-10,TURBO CHARGER,TC-1411649,GC13050098,Kembali
7,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,1350 7712,2,CC 201 77 12,2026-01-12,TURBO CHARGER,TC-1129458,GC 14070087,Kembali
8,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,1341 1511,1,CC 206 13 41,2026-01-12,TURBO CHARGER,TC-1132494,GC13020062,Kembali
9,2026,Daftar Nomor Equipment Lokomotif 2026.xlsx,1341 1511,2,CC 206 15 11,2026-01-16,TURBO CHARGER,TC-1132758,GC13020064,EX CC 206 13 03


## 54. Availability langsung dari database

In [110]:
with closing(get_connection()) as conn:
    db_availability = pd.read_sql_query(
        """
        SELECT source_year,
               COUNT(*) AS sheets,
               SUM(CASE WHEN status = 'OK' THEN 1 ELSE 0 END) AS sheets_ok,
               SUM(block_count) AS blocks,
               SUM(template_block_count) AS blocks_template,
               SUM(event_count) AS events,
               SUM(component_count) AS komponen
        FROM sheet_availability
        GROUP BY source_year
        ORDER BY source_year;
        """,
        conn
    )

display(db_availability)

with closing(get_connection()) as conn:
    db_tanggal = pd.read_sql_query(
        """
        SELECT tahun_maintenance,
               SUM(CASE WHEN masuk_source = 'excel' THEN 1 ELSE 0 END) AS masuk_excel,
               SUM(CASE WHEN masuk_source = 'program_bulan' THEN 1 ELSE 0 END) AS masuk_program,
               SUM(CASE WHEN masuk_source IS NULL THEN 1 ELSE 0 END) AS masuk_kosong,
               SUM(CASE WHEN keluar_source IS NULL THEN 1 ELSE 0 END) AS keluar_kosong,
               COUNT(*) AS events
        FROM maintenance_events
        GROUP BY tahun_maintenance
        ORDER BY tahun_maintenance;
        """,
        conn
    )

print("Asal tanggal per tahun maintenance")

display(db_tanggal)

,source_year,sheets,sheets_ok,blocks,blocks_template,events,komponen
0,2019,36,36,72,0,72,4388
1,2020,34,34,68,1,67,4003
2,2021,41,39,80,2,78,4777
3,2022,55,52,110,6,104,6421
4,2023,56,54,112,5,107,6617
5,2024,71,63,142,17,125,7885
6,2025,69,62,138,16,122,7636
7,2026,54,48,108,12,96,7324


Asal tanggal per tahun maintenance


,tahun_maintenance,masuk_excel,masuk_program,masuk_kosong,keluar_kosong,events
0,2019,0,70,2,2,72
1,2020,55,12,0,0,67
2,2021,78,0,0,1,78
3,2022,103,0,1,0,104
4,2023,107,0,0,0,107
5,2024,125,0,0,0,125
6,2025,122,0,0,0,122
7,2026,96,0,0,20,96


## 55. Ingestion history

In [111]:
with closing(get_connection()) as conn:
    history_df = pd.read_sql_query(
        """
        SELECT id,
               source_file,
               source_year,
               schema_version,
               sheet_count,
               event_count,
               component_count,
               status,
               error_message,
               started_at
        FROM ingestion_history
        ORDER BY id DESC
        LIMIT 20;
        """,
        conn
    )

display(history_df)

,id,source_file,source_year,schema_version,sheet_count,event_count,component_count,status,error_message,started_at
0,24,Daftar Nomor Equipment Lokomotif 2026.xlsx,2026,3,54,96,7324,SUCCESS,None,2026-09-09T08:28:41.996216+00:00
1,23,Daftar Nomor Equipment Lokomotif 2025.xlsx,2025,3,69,122,7636,SUCCESS,None,2026-09-09T08:28:40.394788+00:00
2,22,Daftar Nomor Equipment Lokomotif 2024.xlsx,2024,3,71,125,7885,SUCCESS,None,2026-09-09T08:28:38.752954+00:00
3,21,Daftar Nomor Equipment Lokomotif 2023.xlsx,2023,3,56,107,6617,SUCCESS,None,2026-09-09T08:28:37.281390+00:00
4,20,Daftar Nomor Equipment Lokomotif 2022.xlsx,2022,3,55,104,6421,SUCCESS,None,2026-09-09T08:28:35.837680+00:00
5,19,Daftar Nomor Equipment Lokomotif 2021.xlsx,2021,3,41,78,4777,SUCCESS,None,2026-09-09T08:28:34.762691+00:00
6,18,Daftar Nomor Equipment Lokomotif 2020.xlsx,2020,3,34,67,4003,SUCCESS,None,2026-09-09T08:28:33.862569+00:00
7,17,Daftar Nomor Equipment Lokomotif 2019.xlsx,2019,3,36,72,4388,SUCCESS,None,2026-09-09T08:28:32.848442+00:00
8,16,Daftar Nomor Equipment Lokomotif 2026.xlsx,2026,3,54,96,7324,SUCCESS,None,2026-09-09T08:25:58.244630+00:00
9,15,Daftar Nomor Equipment Lokomotif 2025.xlsx,2025,3,69,122,7636,SUCCESS,None,2026-09-09T08:25:56.877875+00:00


# PART K - Export hasil normalisasi

Empat sheet: event, komponen, availability, dan ringkasan per tahun. Berguna untuk cross check manual sebelum database dipakai lebih lanjut.

Kalau file tujuan sedang dibuka di Excel, Windows mengunci file itu dan penulisan akan gagal. Supaya satu cell terakhir tidak menggagalkan seluruh run, penulisan dialihkan ke nama file bertanda waktu.

In [112]:
EXPORT_EVENT_COLUMNS = [
    "source_file",
    "source_year",
    "source_sheet",
    "block_index",
    "no_seri_lokomotif",
    "lokomotif_no",
    "lokomotif_key",
    "dipo_induk",
    "jenis_perawatan",
    "program_bulan",
    "masuk",
    "keluar",
    "masuk_source",
    "keluar_source",
    "tahun_maintenance",
    "component_count",
]

def write_debug_workbook(path):
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        events_df[EXPORT_EVENT_COLUMNS].to_excel(
            writer,
            sheet_name="maintenance_events",
            index=False
        )

        components_df.to_excel(
            writer,
            sheet_name="equipment_components",
            index=False
        )

        availability_df.to_excel(
            writer,
            sheet_name="sheet_availability",
            index=False
        )

        availability_by_year.reset_index().to_excel(
            writer,
            sheet_name="ringkasan_per_tahun",
            index=False
        )

    return path


try:
    target = write_debug_workbook(DEBUG_OUTPUT)
except PermissionError:
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    fallback = DEBUG_OUTPUT.with_name(
        f"{DEBUG_OUTPUT.stem}_{stamp}{DEBUG_OUTPUT.suffix}"
    )

    print(f"{DEBUG_OUTPUT.name} sedang dibuka di Excel, dialihkan.")

    target = write_debug_workbook(fallback)

print("Debug file:", target.resolve())
print("Events    :", len(events_df))
print("Komponen  :", len(components_df))

Debug file: C:\Users\bnurhuda\Documents\personal\KAI\parsed_lokomotif_debug.xlsx
Events    : 771
Komponen  : 49051


# Catatan

**Aturan data yang dipakai**

- Block tanpa nomor lokomotif dianggap form template dan tidak masuk database, berikut komponennya.
- Row komponen tanpa komponen asal tetapi punya komponen pengganti adalah pemasangan baru, dan tetap disimpan.
- Kalau `MASUK` / `KELUAR` kosong, tanggal diturunkan dari `PROGRAM BULAN`: masuk tanggal 1, keluar tanggal akhir bulan. Kolom `masuk_source` / `keluar_source` menandai mana yang tertulis di Excel dan mana yang diturunkan.
- `tahun_maintenance` diambil dari tahun `masuk`, mundur ke `keluar`, lalu ke tahun file.

**Asumsi struktur yang masih dipegang**

- Urutan 7 kolom detail komponen tetap: `NO.`, `NAMA KOMPONEN`, asal `KODE CETAK` / `NO.MANUF`, pengganti `KODE CETAK` / `NO.MANUF`, `KET.`
- Baris subheader selalu tepat satu baris di bawah baris header.
- Nomor seri lokomotif berada di kanan labelnya, dalam block yang sama.

**Yang tidak dipegang**

- Nomor baris absolut. Semua posisi dicari lewat marker, sehingga hidden row, baris kosong tambahan, dan pergeseran vertikal tidak merusak parsing.
- Nama sheet. Penamaan berbeda-beda antar tahun dan tidak dipakai sebagai identitas.
- Teks label. Varian antar tahun dipetakan lewat `METADATA_LABELS`.

**Kalau ada file tahun baru**

Cukup letakkan di folder yang sama dengan pola nama yang sama, lalu jalankan ulang notebook. File akan terdeteksi otomatis, dan ingestion bersifat idempoten sehingga data lama tidak tergandakan.

**Kalau ada label baru**

Tambahkan variannya pada `METADATA_LABELS`. Cara cepat menemukan varian yang belum tertangani: lihat `sheet_availability` yang berstatus selain `OK`, lalu buka sheet yang tercatat di situ.